# Overview

Document Question Answering, also referred to as Document Visual Question Answering, is a task that involes providing answers to questions posed about document images. The input to models supporting this task is typically a combination of an image and s question, and the output is an answer expressed in natural language. These models utilize multiple modalities, including text, the positions of words(bounding boxes), and the image itself. Here we are going to fine-tune a LLLM [Multi-model Pre-training for Visually-Rich Document Understanding](https://arxiv.org/abs/2012.14740) on the `document-question-answering` dataset. All the models in `LayoutLM*` architectures can be supported.

In [ ]:
%%capture
!pip install transformers==4.35.2
!pip install datasets==2.15.0
!pip install evaluate==0.4.1
# Seqeval is useful for evaluation metrics such as F1 sequence labeling tasks.
!pip install seqeval==1.2.2

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

login(token=user_secrets.get_secret("HUGGINGFACE_TOKEN"))

os.environ["WANDB_API_KEY"]=user_secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "Fine-tuning ms-layoutlm-v3"
os.environ["WANDB_NAME"] = "ft-ms-layoutlmv3-funsd-layoutlmv3"

# Load the data

We will use a small sample of preprocessed DocVQA in here.

In [ ]:
from datasets import load_dataset

dataset=load_dataset("aisuko/funsd-layoutlmv3")
dataset

In [ ]:
dataset["train"].features

In [ ]:
example=dataset["train"][0]
example["image"]

In [ ]:
words, boxes, ner_tags=example["tokens"], example["bboxes"], example["ner_tags"]
print(words)
print(boxes)
print(ner_tags)

# Preparing

Here we prepare the dataset for the model by using `LayoutLMv3Processor`. It internally wraps a `LayoutLMv3FeatureExtractor` for the image modality and a `LayoutLMv3Tokenizer` for the text modality into one. Basically, the processor does the following internally:

- the feature extractor is used to resize+normalize each document image into `pixel_values`
- the tokenizer is ued to turn the words, boxes and NER tags into token-level
  - `input_ids`
  - `attention_mask`
  - `labels`

In [ ]:
from transformers import AutoProcessor

# we'll use the Auto API here - it will load LayoutLMv3Processor behind the scenes,
# based on the checkpoint we provide from the hub
processor=AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
print(processor)

We'll first create `idlabel` and `label2id` mappings, useful for inference.
Note: that `LayoutLMv3ForTokenClassification` (the model we will use later on) will simply output an integer index for a particular class (for each token), so we still need to map it to an actual class name.

In [ ]:
from datasets.features import ClassLabel

features=dataset["train"].features
column_names=dataset["train"].column_names
image_column_name="image"
text_column_name="tokens"
boxes_column_name="bboxes"
label_column_name="ner_tags"

# In the event the labels are not a `Sequence[ClassLabel]`, we will need to go through the dataset to get the unique labels.
def get_label_list(labels):
    unique_labels=set()
    for label in labels:
        unique_labels=unique_labels|set(label)
    label_list=list(unique_labels)
    label_list.sort()
    return label_list

if isinstance(features[label_column_name].feature, ClassLabel):
    label_list=features[label_column_name].feature.names
    # No need to convert the labels since they are already ints
    id2label={k: v for k,v in enumerate(label_list)}
    label2id={v: k for k,v in enumerate(label_list)}
else:
    label_list=get_label_list(dataset["train"][label_column_name])
    id2label={k:v for k,v in enumerate(label_list)}
    label2id={v:k for k,v in enumerate(label_list)}

num_labels=len(label_list)
print(num_labels)

In [ ]:
from datasets import Features, Sequence, ClassLabel, Value, Array2D, Array3D

# we need to define custom features for `set_format`(used later on) to work properly
features=Features({
    'pixel_values': Array3D(dtype="float32", shape=(3,224,224)),
    'input_ids': Sequence(feature=Value(dtype='int64')),
    'attention_mask': Sequence(Value(dtype='int64')),
    'bbox': Array2D(dtype="int64", shape=(512, 4)),
    'labels': Sequence(feature=Value(dtype='int64')),
})


def prepare_examples(examples):
    images=examples[image_column_name]
    words=examples[text_column_name]
    boxes=examples[boxes_column_name]
    word_labels=examples[label_column_name]
    encoding=processor(
        images,
        words,
        boxes=boxes,
        word_labels=word_labels,
        truncation=True,
        padding="max_length"
    )
    return encoding

train_dataset=dataset["train"].map(
    prepare_examples,
    batched=True,
    remove_columns=column_names,
    features=features,
)

eval_dataset=dataset["test"].map(
    prepare_examples,
    batched=True,
    remove_columns=column_names,
    features=features,
)

print(train_dataset)

In [ ]:
example=train_dataset[0]
processor.tokenizer.decode(example["input_ids"])

In [ ]:
train_dataset.set_format("torch")

Let's verify that everying was created properly:

In [ ]:
import torch

example=train_dataset[0]
for k,v in example.items():
    print(k,v.shape)

In [ ]:
processor.tokenizer.decode(eval_dataset[0]["input_ids"])

In [ ]:
for id, label in zip(train_dataset[0]["input_ids"], train_dataset[0]["labels"]):
    print(processor.tokenizer.decode([id]), label.item())

# Define metrics

Let's define a `compute_metrics` function, which is used by the Trainer.

In [ ]:
import numpy as np
import evaluate

return_entity_level_metrics=False
metric=evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels=p
    predictions=np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions=[
        [label_list[p] for (p,l) in zip(prediction, label) if l!=-100]
        for prediction, label in zip(predictions, labels)
    ]
    
    true_labels=[
        [label_list[l] for (p, l) in zip(prediction, label) if l!=-100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results=metric.compute(predictions=true_predictions, references=true_labels)
    
    if return_entity_level_metrics:
        # Unpack nested dictionaries
        final_results={}
        for key, value in results.items():
            if isinstance(value, dict):
                for n,v in value.items():
                    final_results[f"{key}_{n}"]=v
            else:
                final_results[key]=value
        return final_results
    else:
        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        }

# Define the model

This is a Transformer encoder with pre-trained weights, and a randomly initialized head on top for token classification.

In [ ]:
from transformers import LayoutLMv3ForTokenClassification

model=LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers.data.data_collator import default_data_collator


training_args=TrainingArguments(
    output_dir=os.getenv("WANDB_NAME"),
    max_steps=200,
    fp16=True,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-5,
    evaluation_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb", # or report_to="tensorboard"
    run_name=os.getenv("WANDB_NAME"),
    
)


trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model

In [ ]:
trainer.evaluate()

In [ ]:
kwargs={
    'model_name': f'{os.getenv("WANDB_NAME")}',
    'finetuned_from': 'microsoft/layoutlmv3-base',
    'tasks': 'Token Classification',
#     'dataset_tags':'',
    'dataset':'aisuko/funsd-layoutlmv3'
}

processor.push_to_hub(os.getenv("WANDB_NAME"))
trainer.push_to_hub(**kwargs)

# Inference

In [ ]:
from transformers import AutoModelForTokenClassification

model=AutoModelForTokenClassification.from_pretrained(os.getenv("WANDB_NAME"))

In [ ]:
example=dataset["test"][0]
print(example.keys())

In [ ]:
image = example["image"]
words = example["tokens"]
boxes = example["bboxes"]
word_labels = example["ner_tags"]

encoding = processor(image, words, boxes=boxes, word_labels=word_labels, return_tensors="pt")
for k,v in encoding.items():
  print(k,v.shape)

In [ ]:
with torch.no_grad():
    outputs=model(**encoding)

In [ ]:
logits=outputs.logits
logits.shape

We take the highest score for each token, using argmax. This serves as the predicted label for each token.

In [ ]:
predictions=logits.argmax(-1).squeeze().tolist()
print(predictions)

Let's compare this to the ground truth: note that many labels are -100, as we are only labeling the first subword token of each word.

In [ ]:
labels=encoding.labels.squeeze().tolist()
print(labels)

Let's only compare predictions and labels at positions where the label is not -100. 

In [ ]:
def unnormalize_box(bbox, width, height):
     return [
         width * (bbox[0] / 1000),
         height * (bbox[1] / 1000),
         width * (bbox[2] / 1000),
         height * (bbox[3] / 1000),
     ]

token_boxes = encoding.bbox.squeeze().tolist()
width, height = image.size

true_predictions = [model.config.id2label[pred] for pred, label in zip(predictions, labels) if label != - 100]
true_labels = [model.config.id2label[label] for prediction, label in zip(predictions, labels) if label != -100]
true_boxes = [unnormalize_box(box, width, height) for box, label in zip(token_boxes, labels) if label != -100]

In [ ]:
from PIL import ImageDraw, ImageFont

draw = ImageDraw.Draw(image)

font = ImageFont.load_default()

def iob_to_label(label):
    label = label[2:]
    if not label:
        return 'other'
    return label

label2color = {'question':'blue', 'answer':'green', 'header':'orange', 'other':'violet'}

for prediction, box in zip(true_predictions, true_boxes):
    predicted_label = iob_to_label(prediction).lower()
    draw.rectangle(box, outline=label2color[predicted_label])
    draw.text((box[0] + 10, box[1] - 10), text=predicted_label, fill=label2color[predicted_label], font=font)

image

In [ ]:
image = example["image"]
image = image.convert("RGB")

draw = ImageDraw.Draw(image)

for word, box, label in zip(example['tokens'], example['bboxes'], example['ner_tags']):
    actual_label = iob_to_label(id2label[label]).lower()
    box = unnormalize_box(box, width, height)
    draw.rectangle(box, outline=label2color[actual_label], width=2)
    draw.text((box[0] + 10, box[1] - 10), actual_label, fill=label2color[actual_label], font=font)

image